# 08. Reproducible Preprocessing Pipelines & Determinism

### Historical Lineage & Theoretical Foundations
- **Lineage:** *IEEE 754 (1985, Standard for Floating-Point Arithmetic); Kahan (1965, Compensated Summation); Fowler–Noll–Vo (1991, FNV-1a Hash)*.
- **Core Purpose:** Bit-exact hardware reproducibility across training and inference, high-precision Kahan compensated summation to eliminate parallel floating-point non-associativity, and deterministic cryptographic hash partitioning for leakage-free dataset splits.

### Decision Boundary: When NOT to Use
> [!WARNING]
> - **DO NOT rely on unseeded PRNGs or process-dependent shuffling:** Concurrent worker threads and unseeded generators cause non-deterministic partition drift and silent train/validation leakage.
> - **DO NOT use naive summation reductions over long sequences:** Standard reduction accumulates $\mathcal{O}(N \epsilon)$ rounding error; use Kahan compensated summation or deterministic reduction trees when exact numerical parity across runs is required.
> - **DO NOT assume CPU and GPU floating-point operations yield identical bitwise representations:** Different hardware microarchitectures implement FMA (Fused Multiply-Add) contraction and reduction trees differently without hardware-level deterministic algorithmic mode enabled.


### Mathematical Formulation & Asymptotic Complexity

#### 1. Kahan Compensated Summation Formulation
$$y_i = x_i - c_{i-1}, \quad t_i = s_{i-1} + y_i, \quad c_i = (t_i - s_{i-1}) - y_i, \quad s_i = t_i$$
$$\lim_{N \to \infty} \left| s_N - \sum_{i=1}^N x_i \right| \le 2\epsilon \sum_{i=1}^N |x_i| + \mathcal{O}(N \epsilon^2)$$

#### 2. FNV-1a 64-bit Hash Partitioning
$$H_k = (H_{k-1} \oplus B_k) \times P \pmod{2^{64}}, \quad u_{\text{norm}} = \frac{H_K}{2^{64}} \in [0, 1)$$

#### 3. Deterministic Linear Forward Formulation
$$\mathbf{Y} = \mathbf{X} \mathbf{W} + \mathbf{b}$$

**Asymptotic Complexity:**
- Hash Partitioning: Time $\mathcal{O}(L)$ per identifier of byte length $L$; Space $\mathcal{O}(1)$.
- Kahan Summation: Time $\mathcal{O}(B \cdot d)$ with 4 FLOPs per accumulation; Space $\mathcal{O}(d)$ running accumulators.
- Deterministic Linear: Time $\mathcal{O}(B \cdot d \cdot d_{\text{out}})$; Space $\mathcal{O}(B \cdot d_{\text{out}})$.


In [ ]:
from pathlib import Path

import torch
import torch.nn as nn

import core

env = core.setup_env(seed=42)
ckpt_path = Path("weights/00_foundations/08_reproducible_preprocessing_pipelines_and_determinism.safetensors")
if not ckpt_path.is_file():
    ckpt_path = Path("../../weights/00_foundations/08_reproducible_preprocessing_pipelines_and_determinism.safetensors")
weights, test_in, golden_out, meta = core.parity.load_reference_checkpoint(ckpt_path)
dim = int(meta.get("dim", 4))
print(f"Device: {env.device} | Dim: {dim} | Checkpoint: {ckpt_path.name}")


In [ ]:
def kahan_sum(x: torch.Tensor, dim: int = 0) -> torch.Tensor:
    slices = [s.squeeze(dim) for s in torch.split(x, 1, dim=dim)]
    total = torch.zeros_like(slices[0])
    c = torch.zeros_like(slices[0])
    for xi in slices:
        y = xi - c
        t = total + y
        c = (t - total) - y
        total = t
    return total

class ReproducibleLinearPipeline(nn.Module):
    def __init__(self, in_features: int, out_features: int) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_features, out_features))
        self.bias = nn.Parameter(torch.empty(out_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.weight + self.bias

model = ReproducibleLinearPipeline(dim, 2)
with torch.no_grad():
    model.weight.copy_(weights["weight"])
    model.bias.copy_(weights["bias"])
print("ReproducibleLinearPipeline initialized with reference parameters.")


In [ ]:
# 1. Deterministic Multi-Run Bitwise Identity Invariant
out_run1 = model(test_in)
out_run2 = model(test_in)
assert torch.equal(out_run1, out_run2), "Non-deterministic output detected across runs"

# 2. Golden Forward Parity vs Reference Checkpoint
torch.testing.assert_close(out_run1, golden_out, atol=1e-5, rtol=1e-5)

# 3. Kahan Compensated Mean vs Reference
mean_kahan = kahan_sum(test_in, dim=0) / float(test_in.shape[0])
torch.testing.assert_close(mean_kahan, weights["kahan_mean"], atol=1e-6, rtol=1e-6)

# 4. Autograd Gradient Parity
x_param = test_in.clone().detach().requires_grad_(True)
loss = 0.5 * torch.sum((model(x_param) - weights["target"]) ** 2)
loss.backward()
torch.testing.assert_close(model.weight.grad, weights["grad_weight"], atol=1e-5, rtol=1e-5)
torch.testing.assert_close(model.bias.grad, weights["grad_bias"], atol=1e-5, rtol=1e-5)
torch.testing.assert_close(x_param.grad, weights["grad_input"], atol=1e-5, rtol=1e-5)
print("All mathematical invariants (Bitwise Identity + Kahan Sum + Gradients) verified.")


In [ ]:
def run_naive():
    return torch.sum(test_in, dim=0)

def run_kahan():
    return kahan_sum(test_in, dim=0)

def run_pipeline():
    return model(test_in)

track_fns = {"Naive Sum": run_naive, "Kahan Compensated Sum": run_kahan, "Linear Pipeline": run_pipeline}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=test_in.shape[0])
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# [!WARNING] PATHOLOGICAL MISUSE & FAILURE MODE: Unseeded PRNG Partition Drift
# 1. Flawed Anti-Pattern: Unseeded random shuffling alters sample assignment across runs
import random

data_ids = [f"sample_{i:04d}" for i in range(16)]
shuffled_a = data_ids.copy()
shuffled_b = data_ids.copy()
random.seed(101)
random.shuffle(shuffled_a)
random.seed(202)
random.shuffle(shuffled_b)

# Active failure assertion proving non-deterministic partition mismatch
mismatches = sum(a != b for a, b in zip(shuffled_a, shuffled_b, strict=True))
assert mismatches > 0, "Unseeded shuffling must yield divergent partitions across runs"
print(f"[FAILURE DEMONSTRATED] Unseeded shuffle produced {mismatches}/16 misaligned partitions!")

# 2. Correct Pattern: Deterministic FNV-1a Hash-Based Partitioning
def fnv1a_partition(sample_id: str, train_ratio: float = 0.8) -> str:
    h = 0xCBF29CE484222325
    prime = 0x100000001B3
    for b in sample_id.encode("utf-8"):
        h = ((h ^ b) * prime) & 0xFFFFFFFFFFFFFFFF
    return "train" if (h / 0xFFFFFFFFFFFFFFFF) < train_ratio else "val"

split_a = [fnv1a_partition(sid) for sid in data_ids]
split_b = [fnv1a_partition(sid) for sid in data_ids]
assert split_a == split_b, "Hash partitioning must be bitwise identical across all runs"
print("[CORRECT] FNV-1a Hash Partitioning verified: 100% invariant across processes and runs.")

# 3. Compilation Verification
ok_exp, msg_exp = core.export.verify_torch_export(model, test_in)
ok_comp, msg_comp = core.export.verify_torch_compile_fullgraph(model, test_in)
assert ok_exp and ok_comp, "Compilation / export failed"
print("[CORRECT] Pipeline compiled with torch.compile(fullgraph=True) and verified.")


## Architectural Topology, Key Takeaways & Curriculum Navigation

### Deterministic Processing Topology
<svg width="650" height="170" xmlns="http://www.w3.org/2000/svg" style="background:#0f172a; border-radius:8px; font-family:system-ui, sans-serif;">
  <defs>
    <marker id="arrow-d" viewBox="0 0 10 10" refX="6" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#38bdf8"/>
    </marker>
  </defs>
  <text x="30" y="25" fill="#94a3b8" font-size="12" font-weight="bold">REPRODUCIBLE PREPROCESSING &amp; DETERMINISTIC EXECUTION</text>
  <rect x="30" y="50" width="130" height="35" rx="6" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="95" y="72" fill="#38bdf8" font-size="11" text-anchor="middle">Sample ID Stream</text>
  <line x1="160" y1="67" x2="220" y2="67" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow-d)"/>
  <rect x="220" y="50" width="170" height="35" rx="6" fill="#1e293b" stroke="#10b981" stroke-width="1.5"/>
  <text x="305" y="72" fill="#10b981" font-size="11" text-anchor="middle">FNV-1a Hash Partition</text>
  <line x1="390" y1="67" x2="450" y2="67" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow-d)"/>
  <rect x="450" y="50" width="160" height="35" rx="6" fill="#1e293b" stroke="#a855f7" stroke-width="1.5"/>
  <text x="530" y="72" fill="#a855f7" font-size="11" text-anchor="middle">Kahan Deterministic Sum</text>
  <rect x="30" y="110" width="580" height="35" rx="6" fill="#0f172a" stroke="#f59e0b" stroke-width="1.5" stroke-dasharray="4 4"/>
  <text x="320" y="132" fill="#f59e0b" font-size="11" text-anchor="middle">Deterministic Forward Projection: Y = X @ W + b (Bit-Exact Across Multi-Runs)</text>
</svg>

### Key Takeaways
1. **Floating-Point Associativity:** Standard parallel reductions vary with thread scheduling; Kahan compensated summation recovers lost bits and guarantees stable numerical convergence.
2. **Hash-Based Partitioning:** Never rely on global PRNG state for train/test splits; uniform 64-bit hashing guarantees deterministic assignment invariant to worker count and execution order.
3. **Reproducibility Discipline:** Always configure environment determinism via `torch.use_deterministic_algorithms(True)` and fixed seeds across all frameworks.

---
**Curriculum Navigation:**  
&larr; [Previous: 07. Feature Engineering, Scalers & Tabular Preprocessing](07_feature_engineering_scalers_and_tabular_preprocessing.ipynb) | [Master Index: README.md](../../README.md) | [Next: 09. Nested, Ragged & Specialized Sparse Tensors](09_nested_ragged_and_specialized_sparse_tensors.ipynb) &rarr;
